In [ ]:
"""
log2 fold change: KI vs pooled WT, per APOE genotype
Female hippocampus

Input : drift-corrected, PQN-normalised, log2-transformed data (one row per sample)
Output: 1) table with, for each metabolite and each KI line,
           log2FC, p-value (Welch t-test vs pooled WT, Sidak-adjusted for the 3 KI lines)
           and FDR (Benjamini-Hochberg across metabolites)
        2) heatmap of the log2FC (dot = p < 0.05, star = FDR < 0.05)

Why Welch and not Dunnett: Dunnett uses one SD pooled over all four groups. When one
group is much more spread out than the others (e.g. one outlier mouse), that pooled SD
is too small and Dunnett gives false positives. Welch uses each group's own SD.
Welch + Sidak corresponds to Brown-Forsythe/Welch ANOVA + Dunnett T3 in GraphPad.
The Dunnett p-value and the SDs are still saved, for comparison.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------- settings
INPUT_FILE = r"O:\metabolom\Result\new version result_09.2026\Dataset\DrifftCor_PQN\DriftCor_PQNregion_log2_samples_clean.csv"
# the scatter script reads this table, so keep it at the path set in its INPUT_FILE
OUTPUT_FILE = r"O:\metabolom\Result\new version result_09.2026\Dataset\Log2FC_Heatmap_sactter\log2FC_Hippocampus_Female.csv"
HEATMAP_FILE = r"O:\metabolom\Result\new version result_09.2026\Dataset\Log2FC_Heatmap_sactter\log2FC_Hippocampus_Female_heatmap.png"
ONLY_SIGNIFICANT = False       # True: heatmap shows only metabolites with p < 0.05 in any KI line
COLOR_LIMIT = 0.6              # log2FC at which the colour is darkest
REGION = "Hippocmpus"          # spelled as in the data file
SEX = "F"
APOE_LINES = ["APOE2", "APOE3", "APOE4"]

# whole samples to leave out (same list as in the scatter script)
EXCLUDE_SAMPLES = ["34H_AP3_F_WT_163"]
# single values to leave out: (Sample ID, metabolite column), e.g. a failed peak.
# Fill in the Sample ID of the KI-APOE4 mouse with the very low malate value.
EXCLUDE_VALUES = [
    # ("SAMPLE_ID_HERE", "NEG_MALATE"),
]

# ---------------------------------------------------------------- load data
data = pd.read_csv(INPUT_FILE)
samples = data[(data["Region of brain"] == REGION) & (data["Gender"] == SEX)]
samples = samples[~samples["Sample ID"].isin(EXCLUDE_SAMPLES)].copy()
metabolites = [col for col in data.columns if col.startswith(("POS_", "NEG_"))]

for sample_id, met in EXCLUDE_VALUES:
    hit = samples["Sample ID"] == sample_id
    if not hit.any():
        raise SystemExit(f"EXCLUDE_VALUES: sample {sample_id} not found")
    print(f"Left out {met} of {sample_id} (value {samples.loc[hit, met].iloc[0]:.2f})")
    samples.loc[hit, met] = np.nan

wt = samples[samples["Genotype"] == "WT"]                     # pooled WT
ki = {apoe: samples[(samples["Genotype"] == "KI") & (samples["line of APOE"] == apoe)]
      for apoe in APOE_LINES}

print(f"WT (pooled): n = {len(wt)}")
for apoe, group in ki.items():
    print(f"KI-{apoe}: n = {len(group)}")

# ---------------------------------------------------------------- statistics
results = pd.DataFrame({
    "Metabolite": [m.split("_", 1)[1] for m in metabolites],
    "Mode": [m.split("_", 1)[0] for m in metabolites],
})

# Welch t-test: each KI group vs pooled WT, Sidak-adjusted for the 3 comparisons
# (Dunnett p kept only for comparison)
p_welch = {apoe: [] for apoe in APOE_LINES}
p_dunnett = {apoe: [] for apoe in APOE_LINES}
for m in metabolites:
    w = wt[m].dropna()
    ki_groups = [ki[apoe][m].dropna() for apoe in APOE_LINES]
    p_dun = stats.dunnett(*ki_groups, control=w).pvalue
    for apoe, g, p_d in zip(APOE_LINES, ki_groups, p_dun):
        p = stats.ttest_ind(g, w, equal_var=False).pvalue
        p_welch[apoe].append(1 - (1 - p) ** len(APOE_LINES))
        p_dunnett[apoe].append(p_d)

results["SD WT"] = wt[metabolites].std().values
for apoe in APOE_LINES:
    # data are log2, so the fold change is a difference of means (NaN values are skipped)
    results[f"log2FC KI-{apoe}"] = (ki[apoe][metabolites].mean()
                                    - wt[metabolites].mean()).values
    results[f"p KI-{apoe}"] = p_welch[apoe]
    results[f"FDR KI-{apoe}"] = multipletests(p_welch[apoe], method="fdr_bh")[1]
    results[f"p Dunnett KI-{apoe}"] = p_dunnett[apoe]
    results[f"SD KI-{apoe}"] = ki[apoe][metabolites].std().values

# ---------------------------------------------------------------- save
results = results.round(4)
results.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved {len(results)} metabolites to {OUTPUT_FILE}")

for apoe in APOE_LINES:
    n_p = (results[f"p KI-{apoe}"] < 0.05).sum()
    n_fdr = (results[f"FDR KI-{apoe}"] < 0.05).sum()
    print(f"KI-{apoe}: {n_p} metabolites p < 0.05, {n_fdr} FDR < 0.05")

# significant with Dunnett but not with Welch: driven by unequal SDs, not reported
for apoe in APOE_LINES:
    lost = results[(results[f"p Dunnett KI-{apoe}"] < 0.05) & (results[f"p KI-{apoe}"] >= 0.05)]
    if len(lost):
        print(f"KI-{apoe}, p < 0.05 with Dunnett only: "
              + ", ".join(lost["Metabolite"] + " (" + lost["Mode"] + ")"))

# ---------------------------------------------------------------- heatmap
fc = results[[f"log2FC KI-{apoe}" for apoe in APOE_LINES]].to_numpy()
p = results[[f"p KI-{apoe}" for apoe in APOE_LINES]].to_numpy()
fdr = results[[f"FDR KI-{apoe}" for apoe in APOE_LINES]].to_numpy()
labels = (results["Metabolite"] + " (" + results["Mode"] + ")").to_numpy()

if ONLY_SIGNIFICANT:
    keep = (p < 0.05).any(axis=1)
    fc, p, fdr, labels = fc[keep], p[keep], fdr[keep], labels[keep]

# blue = lower in KI, grey = no change, red = higher in KI
cmap = LinearSegmentedColormap.from_list(
    "blue_grey_red", ["#104281", "#3987e5", "#f0efec", "#e34948", "#8f1d1c"])

fig, ax = plt.subplots(figsize=(7, 0.18 * len(labels) + 1.8))
im = ax.imshow(fc, cmap=cmap, vmin=-COLOR_LIMIT, vmax=COLOR_LIMIT, aspect="auto")

# markers instead of text symbols, so they display the same on every computer
rows, cols = (fdr < 0.05).nonzero()
ax.scatter(cols, rows, marker="*", s=60, color="black", label="FDR < 0.05")
sig_only = (p < 0.05) & (fdr >= 0.05)
rows, cols = sig_only.nonzero()
ax.scatter(cols, rows, marker="o", s=8, color="black", label="p < 0.05")

ax.set_xticks(range(len(APOE_LINES)), [f"KI\n{apoe}" for apoe in APOE_LINES], fontsize=8)
ax.xaxis.tick_top()
ax.set_yticks(range(len(labels)), labels, fontsize=6)
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)

bar = fig.colorbar(im, ax=ax, shrink=0.3, extend="both")
bar.set_label("log2FC vs pooled WT", fontsize=8)
ax.set_title("Female hippocampus: KI vs pooled WT\n(Welch t-test, Sidak-adjusted for 3 KI lines)",
             fontsize=9, pad=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=7, frameon=False)

fig.tight_layout()
fig.savefig(HEATMAP_FILE, dpi=300, bbox_inches="tight")
print(f"Saved heatmap to {HEATMAP_FILE}")
